# 05. 매뉴얼 청크 분할

## 학습 목표

- 페이지 단위 문서를 검색 가능한 청크로 분할한다.
- `chunk_size`, `chunk_overlap`의 의미를 이해한다.
- 페이지, 섹션, 주제 metadata를 구성한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [1]:
from pathlib import Path
import re
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼.pdf")

In [2]:
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def infer_section(page: int) -> str:
    if page <= 4:
        return "민원응대 관련 기본원칙"
    if page <= 7:
        return "일반적인 민원 응대요령"
    if page <= 12:
        return "특이민원 응대요령"
    if page <= 14:
        return "민원담당자 회복 및 보호조치"
    if page <= 17:
        return "질의응답"
    return "참고자료"


def infer_topic(text: str) -> str:
    rules = [
        ("반복전화", ["반복 전화", "반복전화", "동일민원"]),
        ("장시간 통화", ["장시간", "권장시간", "20분", "15분"]),
        ("상급자 통화 요구", ["상급자", "기관장", "부서장"]),
        ("폭언", ["폭언", "욕설", "협박", "성희롱"]),
        ("폭행", ["폭행", "공무집행방해"]),
        ("물품 파손", ["파손", "집기", "공용물건"]),
        ("위험물 소지", ["위험물", "흉기", "119"]),
        ("보호조치", ["휴식", "휴가", "심리상담", "치료"]),
        ("법적 근거", ["형법", "경범죄", "스토킹", "적용법률"]),
    ]

    for topic, keywords in rules:
        if any(keyword in text for keyword in keywords):
            return topic
    return "일반 민원응대"

## 1. 페이지 단위 Document 생성

In [3]:
reader = PdfReader(str(PDF_PATH))
page_docs = []

for page_number, page in enumerate(reader.pages, start=1):
    text = clean_text(page.extract_text() or "")
    if not text:
        continue

    page_docs.append(
        Document(
            page_content=text,
            metadata={
                "source": PDF_PATH.name,
                "page": page_number,
                "section": infer_section(page_number),
                "topic": infer_topic(text)
            }
        )
    )

print("페이지 문서 수:", len(page_docs))

페이지 문서 수: 20


## 2. 청크 분할

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", "STEP", "‣", "•", ". ", " "]
)

chunks = splitter.split_documents(page_docs)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["topic"] = infer_topic(chunk.page_content)

print("청크 수:", len(chunks))

청크 수: 43


## 3. 청크 확인

In [5]:
for chunk in chunks[:5]:
    print("=" * 80)
    print(chunk.metadata)
    print(chunk.page_content[:500])

{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 1, 'section': '민원응대 관련 기본원칙', 'topic': '일반 민원응대', 'chunk_id': 0}
현장공무원을 위한 
민원응대
핵심매뉴얼
발간등록번호
11-1741000-100065-14
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 2, 'section': '민원응대 관련 기본원칙', 'topic': '반복전화', 'chunk_id': 1}
CONTENTS
민원응대 관련 기본원칙 5
일반적인 민원 응대요령 9
특이민원 응대요령 13
1. 공통사항 15
2. 전화응대 15
 2-1 정당한 사유없는 반복전화 15
 2-2 정당한 사유없는 장시간 통화 16
 2-3 상급자(기관장 등) 통화 요구 16
 2-4 폭언(욕설, 협박, 성희롱 등) 17
3. 대면응대 18
 3-1 폭언(욕설, 협박, 성희롱 등) 18
 3-2 폭행 19
 3-3 집기 또는 물품 등을 파손하는 경우 20
 3-4 위험물을 소지하여 신변을 위협하는 경우 21
Ⅰ
Ⅱ
Ⅲ
4. 온라인 민원 및 문서 민원응대 22 
 4-1 폭언(욕설, 협박, 성희롱 등) 22
 4-2 정당한 사유없는 반복민원 23
민원담당자 회복 및 보호조치 25
질의응답 29
[참고1] 특이민원 의미와 유형 35
[참고2] 위법행위 유형별 적용 법률 36
[참고3] 우울증 또는 스트레스 자가진단법 37
Ⅳ
Ⅴ
※ 본 매뉴얼의 민원 응대요령은 민원공무원이 업무 수행 시 참고할 수 있도록
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 3, 'section': '민원응대 관련 기본원칙', 'topic': '일반 민원응대', 'chunk_id': 2}
1. 민원응대 기본 방향 7
2. 행정기관의 민원담당자 보호 7
Ⅰ. 민원응대 관련 기본원칙
Ⅰ 민원응대 관련 기본원칙
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 4, 'section': '민원응대 관련

## 4. 청크 통계 확인

In [8]:
import pandas as pd

chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata["page"],
        "section": doc.metadata["section"],
        "topic": doc.metadata["topic"],
        "length": len(doc.page_content)
    }
    for doc in chunks
])

chunk_df.head()

,chunk_id,page,section,topic,length
0,0,1,민원응대 관련 기본원칙,일반 민원응대,49
1,1,2,민원응대 관련 기본원칙,반복전화,604
2,2,3,민원응대 관련 기본원칙,일반 민원응대,66
3,3,4,민원응대 관련 기본원칙,장시간 통화,697
4,4,4,민원응대 관련 기본원칙,보호조치,226


In [9]:
chunk_df.groupby(["section", "topic"]).size().reset_index(name="count")

,section,topic,count
0,민원담당자 회복 및 보호조치,보호조치,1
1,민원담당자 회복 및 보호조치,상급자 통화 요구,2
2,민원응대 관련 기본원칙,반복전화,1
3,민원응대 관련 기본원칙,보호조치,1
4,민원응대 관련 기본원칙,일반 민원응대,2
5,민원응대 관련 기본원칙,장시간 통화,1
6,일반적인 민원 응대요령,상급자 통화 요구,1
7,일반적인 민원 응대요령,일반 민원응대,3
8,질의응답,물품 파손,1
9,질의응답,반복전화,1


## 핵심 정리

- 청크는 RAG 검색의 최소 단위다.
- 청크가 너무 크면 불필요한 내용이 함께 검색된다.
- 청크가 너무 작으면 문맥이 끊길 수 있다.
- page, section, topic metadata는 검색 결과 해석과 출처 제시에 필요하다.